In [1]:
import os
print("Jalan di:", os.getcwd())
print(".env:", os.path.exists(".env"), "| ca.pem:", os.path.exists("ca.pem"))

Jalan di: C:\Users\ASUS\Downloads\adventure_database
.env: True | ca.pem: True


In [2]:
import os, pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

print("load_dotenv:", load_dotenv())   # harus True

engine = create_engine(
    f"mysql+pymysql://{os.getenv('AIVEN_USER')}:{os.getenv('AIVEN_PASS')}"
    f"@{os.getenv('AIVEN_HOST')}:{os.getenv('AIVEN_PORT')}/{os.getenv('AIVEN_DB')}",
    connect_args={'ssl': {'ca': 'ca.pem'}}
)

load_dotenv: True


In [3]:
berkas = {
    'sales_2020': 'AdventureWorks Sales Data 2020.csv',
    'sales_2021': 'AdventureWorks Sales Data 2021.csv',
    'sales_2022': 'AdventureWorks Sales Data 2022.csv',
    'fact_returns': 'AdventureWorks Returns Data.csv',
    'dim_product': 'AdventureWorks Product Lookup.csv',
    'dim_customer': 'AdventureWorks Customer Lookup.csv',
    'dim_territory': 'AdventureWorks Territory Lookup.csv',
    'dim_calendar': 'AdventureWorks Calendar Lookup.csv',
    'dim_subcategory': 'AdventureWorks Product Subcategories Lookup.csv',
    'dim_category': 'AdventureWorks Product Categories Lookup.csv',
}

def baca_csv(path):
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    return pd.read_csv(path, encoding='latin-1', encoding_errors='replace')

for tabel, path in berkas.items():
    df = baca_csv(path)
    df.to_sql(tabel, engine, if_exists='replace', index=False,
              chunksize=1000, method='multi')
    print(f'{tabel:<15} {len(df):>7} baris terkirim')

sales_2020         2630 baris terkirim
sales_2021        23935 baris terkirim
sales_2022        29481 baris terkirim
fact_returns       1809 baris terkirim
dim_product         293 baris terkirim
dim_customer      18154 baris terkirim
dim_territory        10 baris terkirim
dim_calendar        912 baris terkirim
dim_subcategory      37 baris terkirim
dim_category          4 baris terkirim


In [4]:
from sqlalchemy import text
with engine.connect() as conn:
    for tabel in berkas:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tabel}')).scalar()
        print(f'{tabel:<15} {n:>7}')

sales_2020         2630
sales_2021        23935
sales_2022        29481
fact_returns       1809
dim_product         293
dim_customer      18154
dim_territory        10
dim_calendar        912
dim_subcategory      37
dim_category          4


In [5]:
with engine.connect() as conn:
    for tbl in ["sales_2020", "sales_2021", "sales_2022"]:
        conn.execute(text(
            f"ALTER TABLE {tbl} ADD COLUMN id INT AUTO_INCREMENT PRIMARY KEY FIRST"
        ))
        conn.commit()
        print(f"{tbl}: PK ditambahkan ✓")

sales_2020: PK ditambahkan ✓
sales_2021: PK ditambahkan ✓
sales_2022: PK ditambahkan ✓


In [7]:
from sqlalchemy import inspect

inspector = inspect(engine)
for table in ["sales_2020", "sales_2021", "sales_2022"]:
    pk = inspector.get_pk_constraint(table)['constrained_columns']
    print(f"{table}: PK = {pk}")

sales_2020: PK = ['id']
sales_2021: PK = ['id']
sales_2022: PK = ['id']


In [8]:
from sqlalchemy import inspect

inspector = inspect(engine)
for table in inspector.get_table_names():
    pk = inspector.get_pk_constraint(table)['constrained_columns']
    print(f"{table:20} | PK: {pk if pk else 'BELUM ADA'}")

dim_calendar         | PK: BELUM ADA
dim_category         | PK: BELUM ADA
dim_customer         | PK: BELUM ADA
dim_product          | PK: BELUM ADA
dim_subcategory      | PK: BELUM ADA
dim_territory        | PK: BELUM ADA
fact_returns         | PK: BELUM ADA
sales_2020           | PK: ['id']
sales_2021           | PK: ['id']
sales_2022           | PK: ['id']


In [9]:
with engine.connect() as conn:
    for table in inspector.get_table_names():
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        flag = "⚠️ KENA WARNING" if count > 5000 else "aman"
        print(f"{table:20} | {count:6} baris | {flag}")

dim_calendar         |    912 baris | aman
dim_category         |      4 baris | aman
dim_customer         |  18154 baris | ⚠️ KENA WARNING
dim_product          |    293 baris | aman
dim_subcategory      |     37 baris | aman
dim_territory        |     10 baris | aman
fact_returns         |   1809 baris | aman
sales_2020           |   2630 baris | aman
sales_2021           |  23935 baris | ⚠️ KENA WARNING
sales_2022           |  29481 baris | ⚠️ KENA WARNING


In [10]:
with engine.connect() as conn:
    dup = conn.execute(text(
        "SELECT CustomerKey, COUNT(*) c FROM dim_customer "
        "GROUP BY CustomerKey HAVING c > 1"
    )).fetchall()
    nulls = conn.execute(text(
        "SELECT COUNT(*) FROM dim_customer WHERE CustomerKey IS NULL"
    )).scalar()
    print("Duplikat CustomerKey:", dup if dup else "gak ada")
    print("NULL CustomerKey:", nulls)

Duplikat CustomerKey: [('30---', 3)]
NULL CustomerKey: 1


In [12]:
with engine.connect() as conn:
    for row in conn.execute(text("DESCRIBE dim_customer")):
        if row[0] == "CustomerKey":
            print(row)

('CustomerKey', 'text', 'YES', '', None, '')


In [13]:
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dim_customer WHERE CustomerKey IS NULL"))
    conn.commit()
    print("Baris NULL dihapus ✓")

Baris NULL dihapus ✓


In [14]:
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dim_customer WHERE CustomerKey = '30---'"))
    conn.commit()
    print("Baris '30---' dihapus ✓")

Baris '30---' dihapus ✓


In [16]:
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dim_customer WHERE CustomerKey LIKE 'Export date%'"))
    conn.commit()
    print("Baris 'Export date' dihapus ✓")

Baris 'Export date' dihapus ✓


In [19]:
with engine.connect() as conn:
    aneh = conn.execute(text(
        "SELECT DISTINCT CustomerKey FROM dim_customer "
        "WHERE CustomerKey NOT REGEXP '^[0-9]+$'"
    )).fetchall()
    print("Nilai non-angka tersisa:", aneh if aneh else "gak ada, bersih ✓")

Nilai non-angka tersisa: [('Source AW_Cust_Master',)]


In [21]:
with engine.connect() as conn:
    result = conn.execute(text(
        "DELETE FROM dim_customer WHERE CustomerKey NOT REGEXP '^[0-9]+$'"
    ))
    conn.commit()
    print(f"{result.rowcount} baris non-angka dihapus ✓")

1 baris non-angka dihapus ✓


In [22]:
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE dim_customer MODIFY CustomerKey INT"))
    conn.commit()
    print("CustomerKey diubah ke INT ✓")

CustomerKey diubah ke INT ✓


In [23]:
with engine.connect() as conn:
    # verifikasi terakhir: gak ada duplikat & null
    dup = conn.execute(text(
        "SELECT CustomerKey, COUNT(*) c FROM dim_customer GROUP BY CustomerKey HAVING c > 1"
    )).fetchall()
    nulls = conn.execute(text("SELECT COUNT(*) FROM dim_customer WHERE CustomerKey IS NULL")).scalar()
    print("Duplikat:", dup if dup else "gak ada")
    print("NULL:", nulls)

Duplikat: gak ada
NULL: 0


In [24]:
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE dim_customer ADD PRIMARY KEY (CustomerKey)"))
    conn.commit()
    print("dim_customer: PK (CustomerKey) ✓")

dim_customer: PK (CustomerKey) ✓


In [5]:
from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW fact_sales AS
        SELECT * FROM sales_2020
        UNION ALL
        SELECT * FROM sales_2021
        UNION ALL
        SELECT * FROM sales_2022
    """))

# verifikasi view
with engine.connect() as conn:
    hasil = conn.execute(text("""
        SELECT COUNT(*) AS total, MIN(OrderDate) AS awal, MAX(OrderDate) AS akhir
        FROM fact_sales
    """)).fetchone()
    print("Total baris:", hasil[0], "| Rentang:", hasil[1], "->", hasil[2])

Total baris: 56046 | Rentang: 2020-01-01 -> 2022-06-30


In [7]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    # cek total baris vs jumlah CustomerKey unik
    total = conn.execute(text("SELECT COUNT(*), COUNT(DISTINCT CustomerKey) FROM dim_customer")).fetchone()
    print("Total baris:", total[0], "| CustomerKey unik:", total[1])

    # tampilkan CustomerKey yang muncul lebih dari sekali
    dup = pd.read_sql(text("""
        SELECT CustomerKey, COUNT(*) AS jml
        FROM dim_customer
        GROUP BY CustomerKey
        HAVING COUNT(*) > 1
        ORDER BY jml DESC
        LIMIT 20
    """), conn)
    print(dup)

Total baris: 18154 | CustomerKey unik: 18151
  CustomerKey  jml
0       30---    3


In [8]:
from sqlalchemy import text
cek = {
    'dim_product':  'ProductKey',
    'dim_customer': 'CustomerKey',
    'dim_territory':'TerritoryKey',
}
with engine.connect() as conn:
    for tbl, key in cek.items():
        r = conn.execute(text(f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM {tbl}")).fetchone()
        status = "OK unik" if r[0]==r[1] else f"DUPLIKAT ({r[0]-r[1]} baris)"
        print(f"{tbl:<15} {key:<13} total={r[0]:<6} unik={r[1]:<6} -> {status}")

dim_product     ProductKey    total=293    unik=293    -> OK unik
dim_customer    CustomerKey   total=18154  unik=18151  -> DUPLIKAT (3 baris)


OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'TerritoryKey' in 'field list'")
[SQL: SELECT COUNT(*), COUNT(DISTINCT TerritoryKey) FROM dim_territory]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [9]:
from sqlalchemy import text
for tbl in ['dim_territory', 'dim_calendar', 'fact_returns', 'fact_sales']:
    cols = pd.read_sql(text(f"SELECT * FROM {tbl} LIMIT 0"), engine).columns.tolist()
    print(f"{tbl}:")
    print("   ", cols)

dim_territory:
    ['SalesTerritoryKey', 'Region', 'Country', 'Continent']
dim_calendar:
    ['Date']
fact_returns:
    ['ReturnDate', 'TerritoryKey', 'ProductKey', 'ReturnQuantity']
fact_sales:
    ['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']


In [10]:
from sqlalchemy import text
with engine.connect() as conn:
    r = conn.execute(text("SELECT COUNT(*), COUNT(DISTINCT SalesTerritoryKey) FROM dim_territory")).fetchone()
    print("dim_territory:", r[0], "baris,", r[1], "unik ->", "OK" if r[0]==r[1] else "DUPLIKAT")
    # sekalian cek dim_calendar
    r2 = conn.execute(text("SELECT COUNT(*), COUNT(DISTINCT Date) FROM dim_calendar")).fetchone()
    print("dim_calendar:", r2[0], "baris,", r2[1], "unik ->", "OK" if r2[0]==r2[1] else "DUPLIKAT")

dim_territory: 10 baris, 10 unik -> OK
dim_calendar: 912 baris, 912 unik -> OK


In [11]:
import pandas as pd
from sqlalchemy import text

df = pd.read_sql(text("SELECT AnnualIncome FROM dim_customer WHERE AnnualIncome IS NOT NULL"), engine)
print("Statistik AnnualIncome:")
print(df['AnnualIncome'].describe())
print("\nNilai unik (income di AdventureWorks biasanya kelipatan rapi):")
print(sorted(df['AnnualIncome'].unique()))

Statistik AnnualIncome:
count     18148.000000
mean      57269.120564
std       32236.535573
min       10000.000000
25%       30000.000000
50%       60000.000000
75%       70000.000000
max      170000.000000
Name: AnnualIncome, dtype: float64

Nilai unik (income di AdventureWorks biasanya kelipatan rapi):
[np.float64(10000.0), np.float64(20000.0), np.float64(30000.0), np.float64(40000.0), np.float64(50000.0), np.float64(60000.0), np.float64(70000.0), np.float64(80000.0), np.float64(90000.0), np.float64(100000.0), np.float64(110000.0), np.float64(120000.0), np.float64(130000.0), np.float64(150000.0), np.float64(160000.0), np.float64(170000.0)]


In [12]:
from sqlalchemy import inspect

inspector = inspect(engine)

# daftar semua tabel
print(inspector.get_table_names())

# struktur satu tabel
for col in inspector.get_columns('sales_2020'):
    print(col['name'], '|', col['type'], '|', 'nullable:', col['nullable'])

# cek primary key
print("PK:", inspector.get_pk_constraint('sales_2020'))

['dim_calendar', 'dim_category', 'dim_customer', 'dim_product', 'dim_subcategory', 'dim_territory', 'fact_returns', 'sales_2020', 'sales_2021', 'sales_2022']
OrderDate | TEXT | nullable: True
StockDate | TEXT | nullable: True
OrderNumber | TEXT | nullable: True
ProductKey | BIGINT | nullable: True
CustomerKey | BIGINT | nullable: True
TerritoryKey | BIGINT | nullable: True
OrderLineItem | BIGINT | nullable: True
OrderQuantity | BIGINT | nullable: True
PK: {'name': None, 'constrained_columns': []}


In [13]:
from sqlalchemy import text
import pandas as pd


for tbl in berkas.keys():
    cols = pd.read_sql(text(f"SELECT * FROM {tbl} LIMIT 0"), engine).columns.tolist()
    
    print(f"{tbl}:")
    print("   ", cols)
    print("-" * 30)

sales_2020:
    ['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']
------------------------------
sales_2021:
    ['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']
------------------------------
sales_2022:
    ['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']
------------------------------
fact_returns:
    ['ReturnDate', 'TerritoryKey', 'ProductKey', 'ReturnQuantity']
------------------------------
dim_product:
    ['ProductKey', 'ProductSubcategoryKey', 'ProductSKU', 'ProductName', 'ModelName', 'ProductDescription', 'ProductColor', 'ProductSize', 'ProductStyle', 'ProductCost', 'ProductPrice']
------------------------------
dim_customer:
    ['CustomerKey', 'Prefix', 'FirstName', 'LastName', 'BirthDate', 'MaritalStatus', 'Gender', 'EmailAddress', 'AnnualIncome', 'TotalChildre